# Time Series Benchmark Dataset Collection & Moving Average Demo

This demo notebook loads standardized time series benchmark datasets (including synthetic trend-seasonality regimes, abrupt regime shifts, and random walks) from JSON, and evaluates moving average smoothing vs naive last-value forecasting.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'scikit-learn==1.6.1')

In [ ]:
import os
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# NumPy 2.0 compatibility shim if needed
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-230d6e-robust-temporal-smoothing-evaluating-mov/main/round-2/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"Failed to load from GitHub URL ({e}), falling back to local file...")
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local disk.")

data = load_data()
print(f"Loaded {len(data['datasets'])} datasets successfully.")

In [ ]:
# Config parameters
WINDOW_SIZE = 3
MAX_DATASETS = 5

In [ ]:
results = []

for ds in data["datasets"][:MAX_DATASETS]:
    name = ds["dataset"]
    examples = ds["examples"]
    
    # Reconstruct full signal from past values and outputs
    # Each example has input (past_values of length 3) and output (curr_val)
    signal = []
    if len(examples) > 0:
        first_past = json.loads(examples[0]["input"])["past_values"]
        signal.extend(first_past)
        for ex in examples:
            signal.append(float(ex["output"]))
            
    s = pd.Series(signal)
    # Moving average forecast
    ma_pred = s.rolling(window=WINDOW_SIZE).mean().shift(1)
    # Naive last-value forecast
    naive_pred = s.shift(1)
    
    y_true = s.iloc[WINDOW_SIZE:].values
    y_ma = ma_pred.iloc[WINDOW_SIZE:].values
    y_naive = naive_pred.iloc[WINDOW_SIZE:].values
    
    mse_ma = np.mean((y_true - y_ma) ** 2)
    mse_naive = np.mean((y_true - y_naive) ** 2)
    
    results.append({
        "dataset": name,
        "mse_ma": mse_ma,
        "mse_naive": mse_naive,
        "signal_preview": signal[:10]
    })
    print(f"Dataset: {name} | MA{WINDOW_SIZE} MSE: {mse_ma:.4f} | Naive MSE: {mse_naive:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
for res in results:
    plt.plot(res["signal_preview"], label=res["dataset"])
plt.title("Preview of Reconstructed Time Series Signals")
plt.xlabel("Time Step")
plt.ylabel("Value")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("\nSummary Table:")
for res in results:
    print(f"{res['dataset']}: MA MSE={res['mse_ma']:.4f}, Naive MSE={res['mse_naive']:.4f}")